In [ ]:
!pip install optuna optuna_dashboard optuna-integration[tfkeras]

In [ ]:
# Cell 2
from google.colab import drive, runtime
import optuna
drive.mount('/content/drive', force_remount=True)        # ← prompts for OAuth

#  EDIT ME – Point to your project root inside Drive
BASE = '/content/drive/MyDrive/ColabNotebooks/PLACEHOLDER_BASE/'     # ← rename as you like

# Auto-create sub-folders identical to your local workflow
import pathlib, os, json, random, math, shutil, gc, threading, time, io
SEED = 42
PROJECT_NAME = "PLACEHOLDER_PROJECT"  # ← rename as you like
MAX_TRIALS         = 70
SAMPLER = optuna.samplers.TPESampler(multivariate=True, seed=SEED, n_startup_trials=75)
PRUNER = optuna.pruners.HyperbandPruner()


# SAMPLER = optuna.samplers.TPESampler(multivariate=False, seed=SEED, n_startup_trials=75)
# PRUNER = optuna.pruners.HyperbandPruner()

# SAMPLER = optuna.samplers.GPSampler(seed=SEED, n_startup_trials=48)
# PRUNER = optuna.pruners.NopPruner()

# ["swing", "stance", "stride", "cadence"]
EPOCHS, PATIENCE        = 100, 10
TEST_FRAC = 0.10
TARGETS = ["swing", "stance", "stride", "cadence"]
POOLING_MODE = "staggered"  # "staggered" (every 2nd conv or final conv) or "every_conv" (after every conv)
ROOT        = pathlib.Path(BASE)
WORK_DIR    = ROOT / 'PLACEHOLDER_DIR'/ f'{PROJECT_NAME}'                               ; WORK_DIR.mkdir(exist_ok=True, parents=True)
FEATURE_DIR = ROOT / 'Data' / 'YOLOv11m_with_linear_interpolation_and_upper_limbs_removed'     # ← must contain *.csv per clip
LABELS_CSV = ROOT / 'Data' / 'Labels.csv'                   # ← Ground truth labels for each clip

print('✓ Paths set')


In [ ]:
# ── utils_scaler.py ─────────────────────────────────────
import json, numpy as np, pathlib

def fit_scaler(X, method="zscore", groups=None, eps=1e-8):
    """
    X : (N, W, F) ndarray  – complete window tensor from training split
    groups : list[list[int]] | None
        e.g. [[0,1,2], [3,4,5]] to couple (x,y,z) triplets.
        If None ⇒ treat each feature independently.
    Returns a dict {feat_idx: {"mean": ..., "std": ...}}  or {"max": ...}
    """
    F = X.shape[-1]
    if groups is None:
        groups = [[i] for i in range(F)]

    stats = {}
    for g in groups:
        slice_ = X[..., g].reshape(-1, len(g))
        if method == "zscore":
            mu  = slice_.mean(axis=0)
            std = slice_.std(axis=0) + eps
            for i, f in enumerate(g):
                stats[f] = {"mean": float(mu[i]), "std": float(std[i])}
        elif method == "max":
            mx = np.abs(slice_).max(axis=0) + eps
            for i, f in enumerate(g):
                stats[f] = {"max": float(mx[i])}
        else:
            raise ValueError("method must be 'zscore' or 'max'")
    return stats

def apply_scaler(X, stats, method="zscore"):
    F = X.shape[-1]
    for f in range(F):
        if method == "zscore":
            X[..., f] = (X[..., f] - stats[f]["mean"]) / stats[f]["std"]
        else:  # max-norm
            X[..., f] = X[..., f] / stats[f]["max"]
    return X

def save_stats(stats, path):
    pathlib.Path(path).write_text(json.dumps(stats, indent=2))

def load_stats(path):
    import json
    return json.loads(open(path).read())
# ────────────────────────────────────────────────────────


In [ ]:
# Cell 3
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import layers, Model, callbacks, optimizers, metrics as keras_metrics
from optuna.integration import TFKerasPruningCallback
from optuna.importance import get_param_importances

import os, random, io, time, json, math, shutil, pathlib, threading, gc
from typing import Dict, Any

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

try:
    from optuna_dashboard import run_server as _run_optuna_dashboard
    _HAS_DASH = False
except ImportError:
    _HAS_DASH = False

try:
    # TF 2.11+ (official)
    from tensorflow.keras.optimizers import AdamW
except ImportError:
    # Older TF – needs tensorflow‑addons
    from tensorflow_addons.optimizers import AdamW


# 3) XLA (just-in-time compile kernels)
tf.config.optimizer.set_jit(True)

# 4) Determinism (keep if you need reproducibility)
os.environ["PYTHONHASHSEED"]       = str(SEED)   # A
os.environ["TF_DETERMINISTIC_OPS"] = "1"         # B
random.seed(SEED)                            # C
np.random.seed(SEED)                         # D
tf.keras.utils.set_random_seed(SEED)         # E
tf.config.experimental.enable_op_determinism()   # F  (works in TF‑2.10)

try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
    STRAT = tf.distribute.TPUStrategy()
    print(' Using TPU v3-8')
except ValueError:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for g in gpus: tf.config.experimental.set_memory_growth(g, True)
        STRAT = tf.distribute.MirroredStrategy()
        print(f' Using {len(gpus)} × {gpus[0].name.split("/")[-1]}')
    else:
        STRAT = tf.distribute.get_strategy()
        print(' CPU only')

import keras

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)


In [ ]:
LOSS_FN = tf.keras.losses.MeanSquaredError()  # or MAE/Huber

NORM_METHOD          = "max"    # label normalisation
FEATURE_NORM_METHOD  = "max"    # feature normalisation
COUPLED_GROUPS       = None     # None ⇒ per‑feature scaling

# Optuna budget
EXECUTIONS_PER_TR  = 1        # keep 1 – use pruning instead

RESULTS_CSV   = os.path.join(WORK_DIR, "optuna_results.csv")
BEST_MODEL    = os.path.join(WORK_DIR, "best_model.keras")
EXCEL_OUT     = os.path.join(WORK_DIR, "training_summary.xlsx")
BEST_HP_JSON  = os.path.join(WORK_DIR, "best_hp.json")
STUDY_DB     = os.path.join(WORK_DIR, f"{PROJECT_NAME}.db")

ARCHES = ["conv_bi_lstm"]
# ARCHES = ["bi_lstm", "conv_bi_lstm", "conv_lstm", "lstm"]

# ═════════════════════════════════════════════════════
# 5) DATA – load labels once, helper to generate windows on‑the‑fly
# ═════════════════════════════════════════════════════
df_labels = pd.read_csv(LABELS_CSV)
NORM_STATS: Dict[str, Dict[str, float]] = {}

for t in TARGETS:
    col = df_labels[t].astype(np.float32)
    if NORM_METHOD == "max":
        mx = col.abs().max() or 1.0
        df_labels[t] = col / mx
        NORM_STATS[t] = {"max": float(mx)}
    elif NORM_METHOD == "zscore":
        mu, sig = col.mean(), col.std(ddof=0) or 1.0
        df_labels[t] = (col - mu) / sig
        NORM_STATS[t] = {"mean": float(mu), "std": float(sig)}
    else:
        raise ValueError("Unknown NORM_METHOD")

stats_path = pathlib.Path(WORK_DIR, f"norm_stats_{NORM_METHOD}.json")
stats_path.write_text(json.dumps(NORM_STATS, indent=2))
print(f"✓ wrote normalisation stats → {stats_path}")

df_labels["stratify_key"] = df_labels.get("type", 0)
trainval_df, test_df = train_test_split(
    df_labels, test_size=TEST_FRAC,
    stratify=df_labels["stratify_key"],
    random_state=SEED)

# Save reserved test set filenames for inspection
with open(os.path.join(WORK_DIR, "test_set_filenames.txt"), "w") as f:
    f.writelines(f"{fn}\n" for fn in test_df["filename"].tolist())
print("✓ Test set filenames saved")

# ----------------------------------------------------------------------
# Helper functions (unchanged)
# ----------------------------------------------------------------------

def sliding_window(arr: np.ndarray, w: int, s: int):
    T, _ = arr.shape
    if T < w:
        arr = np.vstack([arr, np.repeat(arr[-1:], w - T, axis=0)])
    return np.stack([arr[i:i + w] for i in range(0, arr.shape[0] - w + 1, s)])


def make_windows_and_labels(df: pd.DataFrame, w: int, s: int):
    X, y = [], []
    for _, r in df.iterrows():
        arr = pd.read_csv(f"{FEATURE_DIR}/{r.filename}.csv").iloc[:, 1:13].values.astype(np.float32)
        for win in sliding_window(arr, w, s):
            if not np.isnan(win).any():
                X.append(win)
                y.append([r[t] for t in TARGETS])
    if not X:
        raise ValueError("No valid windows – check paths / ranges")
    return np.stack(X), np.array(y, np.float32)


def sane_pooling(hp_dict: Dict[str, Any]):
    num_stacks = int(hp_dict["num_cnn_stacks"])
    pool_size  = int(hp_dict["pool_size"])
    if POOLING_MODE == "staggered":
        pools = math.ceil(num_stacks / 2)
    elif POOLING_MODE == "every_conv":
        pools = num_stacks
    else:
        raise ValueError(f"Unknown POOLING_MODE: {POOLING_MODE}")
    tot_pool = pool_size ** pools
    return int(hp_dict["window_size"]) // tot_pool >= 1



# ----------------------------------------------------------------------
# 6)  Model builder (identical to old version except param dict source)
# ----------------------------------------------------------------------
def build_conv_stem(x, hp):
    num_stacks = int(hp["num_cnn_stacks"])
    pool_size  = int(hp["pool_size"])

    if POOLING_MODE not in {"staggered", "every_conv"}:
        raise ValueError(f"Unknown POOLING_MODE: {POOLING_MODE}")

    for i in range(num_stacks):
        x = layers.Conv1D(
            filters=int(hp["cnn_filters"]),
            kernel_size=int(hp["cnn_kernel"]),
            padding="same",
            activation="relu",
            kernel_initializer=tf.keras.initializers.HeNormal(seed=SEED),
        )(x)

        should_pool = (
            (i % 2 == 1) or (i == num_stacks - 1)
            if POOLING_MODE == "staggered"
            else True
        )
        if should_pool:
            x = layers.MaxPooling1D(pool_size=pool_size)(x)

    if hp.get("use_batchnorm", 0):
        x = layers.BatchNormalization()(x)
    if hp["cnn_dropout"] > 0:
        x = layers.Dropout(hp["cnn_dropout"], seed=SEED)(x)
    return x


def lstm_stack(x, hp, bidirectional=False):
    for i in range(int(hp["lstm_layers"])):
        ret_seq = i < int(hp["lstm_layers"]) - 1
        lstm = layers.LSTM(int(hp["lstm_units"]),
                           return_sequences=ret_seq,
                           kernel_initializer=tf.keras.initializers.HeNormal(seed=SEED),
                           recurrent_initializer=tf.keras.initializers.Orthogonal(seed=SEED))
        x = layers.Bidirectional(lstm)(x) if bidirectional else lstm(x)
    return x


def build_model(hp: Dict[str, Any], arch: str = "conv_lstm"):
    ws, nf = int(hp["window_size"]), 12
    inp = x = layers.Input((ws, nf))

    if arch == "conv_lstm":
        x = build_conv_stem(x, hp)
        x = lstm_stack(x, hp, bidirectional=False)
    elif arch == "bi_lstm":
        x = lstm_stack(x, hp, bidirectional=True)
    elif arch == "lstm":
        x = lstm_stack(x, hp, bidirectional=False)
    elif arch == "conv_bi_lstm":
        x = build_conv_stem(x, hp)
        x = lstm_stack(x, hp, bidirectional=True)
    else:
        raise ValueError(f"Unknown arch: {arch}")

    if hp["dropout"] > 0:
        x = layers.Dropout(hp["dropout"], seed=SEED)(x)
    x = layers.Dense(int(hp["dense_units"]), activation="relu",
                     kernel_initializer=tf.keras.initializers.HeNormal(seed=SEED))(x)
    out = layers.Dense(len(TARGETS), activation="linear")(x)

    model = Model(inp, out)

    # optimiser choice
    lr = hp["learning_rate"]
    wd = hp.get("weight_decay", 1e-4)
    opt = AdamW(learning_rate=lr, weight_decay=wd, clipnorm=1.0)

    model.compile(
        opt, loss=LOSS_FN,
        metrics=["mae", "mse",
                 keras_metrics.RootMeanSquaredError(name="rmse"),
                 keras_metrics.MeanAbsolutePercentageError(name="mape")]
    )
    return model

# ----------------------------------------------------------------------
# 7)  Optuna Objective
# ----------------------------------------------------------------------

def _suggest_space(trial: optuna.Trial) -> Dict[str, Any]:
    hp = {
        # data level
        "window_size":  trial.suggest_int("window_size", 70, 250, step=10),
        "stride":       trial.suggest_int("stride", 1, 5),
        "batch_size":   trial.suggest_categorical("batch_size", [32, 64, 128]),
        # CNN
        "num_cnn_stacks": trial.suggest_int("num_cnn_stacks", 1, 3),
        "cnn_filters":    trial.suggest_int("cnn_filters", 32, 128, step=32),
        "cnn_kernel":     trial.suggest_categorical("cnn_kernel", [3, 5, 7]),
        "pool_size":      trial.suggest_categorical("pool_size", [2, 5]),
        "cnn_dropout":    trial.suggest_float("cnn_dropout", 0.0, 0.5, step=0.1),
        "use_batchnorm":  trial.suggest_categorical("use_batchnorm", [0, 1]),
        # LSTM + dense
        "lstm_layers":  trial.suggest_int("lstm_layers", 1, 3),
        "lstm_units":   trial.suggest_int("lstm_units", 32, 128, step=32),
        "dropout":      trial.suggest_float("dropout", 0.1, 0.5, step=0.1),
        "dense_units":  trial.suggest_int("dense_units", 32, 128, step=32),
        # optimiser
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
        "weight_decay":  trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
    }
    return hp


def objective(trial: optuna.Trial):
    hp = _suggest_space(trial)
    arch = ARCHES[0]

    if not sane_pooling(hp):
        raise optuna.exceptions.TrialPruned()

    # ---- dataset ------------------------------------------------------
    ws, st, bs = hp["window_size"], hp["stride"], hp["batch_size"]

    tr_df, val_df = train_test_split(
    trainval_df,
    test_size=0.10,                       # 10 % of files for validation
    stratify=trainval_df["stratify_key"],
    random_state=SEED)

    X_tr,  y_tr  = make_windows_and_labels(tr_df,  ws, st)
    X_val, y_val = make_windows_and_labels(val_df, ws, st)

    scaler = fit_scaler(X_tr, FEATURE_NORM_METHOD, COUPLED_GROUPS)
    X_tr  = apply_scaler(X_tr,  scaler, FEATURE_NORM_METHOD)
    X_val = apply_scaler(X_val, scaler, FEATURE_NORM_METHOD)

    # ---- model --------------------------------------------------------
    model = build_model(hp, arch)

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        batch_size=bs,
        epochs=EPOCHS,
        verbose=0,
        callbacks=[
            TFKerasPruningCallback(trial, "val_rmse"),
            callbacks.EarlyStopping("val_rmse", patience=PATIENCE, restore_best_weights=True),
        ]
    )

    val_rmse = min(history.history["val_rmse"])

    # cleanup to free GPU VRAM
    tf.keras.backend.clear_session()
    gc.collect()

    return val_rmse

# ----------------------------------------------------------------------
# 8)  Run study
# ----------------------------------------------------------------------

def launch_dashboard():
    if not _HAS_DASH:
        print("⚠ optuna-dashboard not installed – skipping live UI")
        return

    url = f"sqlite:///{STUDY_DB}"
    # supply the study name with the correct keyword
    _run_optuna_dashboard(url,
                          host="localhost", port=8080)




def main():
    study = optuna.create_study(
        study_name=PROJECT_NAME,
        direction="minimize",
        sampler=SAMPLER,
        pruner=PRUNER,
        storage=f"sqlite:///{STUDY_DB}",
        load_if_exists=True,
    )

    # dashboard in background
    if _HAS_DASH:
        threading.Thread(target=launch_dashboard, daemon=True).start()
        print("🔎 Optuna dashboard → http://localhost:8080  (Ctrl-C twice to quit)")

    from optuna.trial import TrialState

    def n_completed() -> int:
        """Count only trials that finished successfully."""
        return len(
            study.get_trials(deepcopy=False, states=(TrialState.COMPLETE,))
        )
    t0 = time.time()

    while True:
        done = n_completed()
        print(f"✓ Completed trials so far: {done}")
        if done >= MAX_TRIALS:
            print(f"🏁  Reached {MAX_TRIALS} completed trials – stopping.")
            break
        remaining = MAX_TRIALS - done
        print(f"{done} completed trials logged, launching {remaining} more …")
        study.optimize(objective, n_trials=remaining, show_progress_bar=True)

    # dataframe of all trials
    study.trials_dataframe().to_csv(RESULTS_CSV, index=False)
    print(f"📝 {len(study.trials)} trials logged → {RESULTS_CSV}")

    # importance analysis ------------------------------------------------
    importances = get_param_importances(study)
    imp_path = os.path.join(WORK_DIR, "param_importance.json")
    pathlib.Path(imp_path).write_text(json.dumps(importances, indent=2))
    print(f"★ Parameter importances written → {imp_path}")

    # best params --------------------------------------------------------
    best_hp = study.best_trial.params
    pathlib.Path(BEST_HP_JSON).write_text(json.dumps(best_hp, indent=2))
    print("★ Best HP saved → best_hp.json")

    # ----- retrain best HP on full train‑val ---------------------------
    X_full, y_full = make_windows_and_labels(trainval_df,
                        best_hp["window_size"], best_hp["stride"])
    scaler_full = fit_scaler(X_full, FEATURE_NORM_METHOD, COUPLED_GROUPS)
    save_stats(scaler_full, os.path.join(WORK_DIR,
            f"norm_stats_trainval_{FEATURE_NORM_METHOD}.json"))
    X_full = apply_scaler(X_full, scaler_full, FEATURE_NORM_METHOD)

    arch = ARCHES[0]
    best_model = build_model(best_hp, arch)
    best_model.fit(
        X_full, y_full,
        validation_split=0.1,
        epochs=EPOCHS,
        batch_size=best_hp["batch_size"],
        verbose=2,
        callbacks=[
            callbacks.EarlyStopping("val_rmse", patience=PATIENCE, restore_best_weights=True),
            callbacks.ModelCheckpoint(BEST_MODEL, monitor="val_rmse", save_best_only=True),
            callbacks.ReduceLROnPlateau("val_rmse", factor=0.5, patience=5),
            callbacks.TerminateOnNaN(),
        ]
    )

    # ----- evaluate on hold‑out ----------------------------------------
    final = tf.keras.models.load_model(BEST_MODEL)
    X_test, y_test = make_windows_and_labels(test_df,
                        best_hp["window_size"], best_hp["stride"])
    X_test = apply_scaler(X_test, scaler_full, FEATURE_NORM_METHOD)

    loss, mae, mse, rmse, mape = final.evaluate(X_test, y_test, verbose=0)
    preds = final.predict(X_test, verbose=0)
    r2    = r2_score(y_test, preds)
    print(f"Hold‑out  RMSE={rmse:.5f}  MAE={mae:.5f}  R²={r2:.5f}")

    # ----- Excel summary ----------------------------------------------
    with pd.ExcelWriter(EXCEL_OUT) as w:
        pd.DataFrame([{
            "test_rmse": rmse,
            "test_mae":  mae,
            "test_mse":  mse,
            "test_mape": mape,
            "test_r2":   r2,
            "total_time_s": time.time() - t0,
        }]).to_excel(w, sheet_name="summary", index=False)

        pd.DataFrame(best_hp.items(), columns=["hyperparameter", "value"]).to_excel(
            w, sheet_name="best_hyperparameters", index=False)

        pd.read_csv(RESULTS_CSV).to_excel(w, sheet_name="optuna_trials", index=False)

        pd.DataFrame(importances.items(), columns=["hyperparameter", "importance"\
            ]).to_excel(w, sheet_name="param_importance", index=False)

    print(f"✔ Done – artefacts saved in {WORK_DIR}")


In [ ]:
if __name__ == "__main__":
    main()